<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/07_system_design.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/07_system_design.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# 07 — System Design for ML Platforms

**Objectives**
- Translate product requirements (QPS, latency SLO) into capacity plans.
- Apply Little's law and a queueing approximation to size replicas.
- Build a cost model and expose the tradeoff between headroom and spend.

Exercise: fill in the `requirements` block for your own scenario and re-run.


In [ ]:
import math

requirements = {
    "qps_peak": 120,  # requests per second at peak
    "latency_slo_ms": 300,  # p99 target
    "avg_latency_ms": 120,  # measured single-request latency
    "gpu_throughput_rps": 25,  # requests/sec one replica sustains
    "headroom": 1.4,  # safety factor for bursts/failures
    "gpu_hourly_usd": 3.06,
}

replicas = math.ceil(
    requirements["qps_peak"] * requirements["headroom"] / requirements["gpu_throughput_rps"]
)
print(f"replicas_needed: {replicas}")
print(f"provisioned throughput: {replicas * requirements['gpu_throughput_rps']:.0f} req/s")

In [ ]:
# Little's law: concurrency L = arrival rate (lambda) * time in system (W)
lam = requirements["qps_peak"]
W = requirements["avg_latency_ms"] / 1000
concurrency = lam * W
print(f"concurrent_requests: {concurrency:.1f}")


# M/M/c queue approximation for the probability of queueing
def erlang_c(c: int, offered_load: float) -> float:
    a = offered_load
    top = (a**c) / math.factorial(c) * c / (c - a) if a < c else float("inf")
    bottom = sum((a**k) / math.factorial(k) for k in range(c)) + top
    return top / bottom if bottom else 1.0


per_replica_load = lam / requirements["gpu_throughput_rps"]
p_queue = erlang_c(replicas, per_replica_load)
print(f"utilization_per_replica: {per_replica_load:.2f}")
print(f"probability_of_queueing: {p_queue:.3f}")

In [ ]:
# Cost model: idle headroom is insurance, not waste (until it is)
hours_per_month = 730
monthly_cost = replicas * requirements["gpu_hourly_usd"] * hours_per_month
requests_per_month = requirements["qps_peak"] * 60 * 60 * hours_per_month
cost_per_1k = monthly_cost / (requests_per_month / 1000)

print(f"monthly_gpu_cost_usd: {monthly_cost:,.0f}")
print(f"cost_per_1k_requests_usd: {cost_per_1k:.4f}")
print("SLO review: p99 target", requirements["latency_slo_ms"], "ms")

**Architect's checklist**
1. Capacity must be sized for *peak*, not average; keep an explicit headroom factor.
2. Queueing probability explodes as utilization approaches 1 — the last 10% of
   utilization buys very little throughput and a lot of tail latency.
3. Cost per 1K requests is the metric to compare against business value.
